# GeraÃ§Ã£o das imagens falsas, pareadas por conteÃºdo com as reais
Para cada foto real, o notebook (1) gera uma **legenda sem nomes** com BLIP e (2) gera uma imagem falsa a partir dela, alternando 3 geradores (SDXL, SD 1.5, SDXL-Turbo). Assim as duas classes tÃªm a mesma distribuiÃ§Ã£o de cenas, e cada falsa guarda gerador, prompt e semente.
**Antes:** compacte a pasta `dataset/real_oficial` em `real_oficial.zip` e envie para `MyDrive/tcc_politicos/`. Ligue a GPU T4.
Todas as falsas sÃ£o de **pessoas fictÃ­cias** (nenhum nome de polÃ­tico entra no prompt). Mantenha as imagens geradas privadas, sÃ³ para pesquisa.

In [ ]:
# 1) Drive, GPU e pastas
import os, subprocess, zipfile
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
from google.colab import drive; drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/tcc_politicos"; os.makedirs(ROOT, exist_ok=True)
REAL = "/content/real_oficial"
def unzip_fix(zpath, dest="/content"):   # zip criado no Windows usa "\" nos caminhos; converte para "/"
    with zipfile.ZipFile(zpath) as z:
        for i in z.infolist():
            n = i.filename.replace("\\", "/")
            if n.endswith("/"): continue
            p = os.path.join(dest, n); os.makedirs(os.path.dirname(p), exist_ok=True)
            with open(p, "wb") as f: f.write(z.read(i))
if not os.path.isdir(REAL) or not os.listdir(REAL):
    unzip_fix(f"{ROOT}/real_oficial.zip")
reais = sorted(f for f in os.listdir(REAL) if f.lower().endswith((".jpg", ".jpeg", ".png")))
print(len(reais), "fotos reais")
FAKE = f"{ROOT}/fake_geradas"; os.makedirs(FAKE, exist_ok=True)

In [ ]:
# 2) InstalaÃ§Ã£o
import subprocess
print(subprocess.run("pip -q uninstall -y torchao 2>&1 | tail -1; pip -q install -U diffusers transformers accelerate safetensors 2>&1 | tail -1", shell=True, capture_output=True, text=True).stdout)
print("Se esta celula acabou de instalar algo, use Ambiente de execucao > Reiniciar sessao e rode de novo as celulas 1 e 4 (as legendas ja ficam salvas).")

In [ ]:
# 3) Legendas das reais com BLIP (sem nomes). Salva em legendas.csv
import torch, pandas as pd
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from tqdm.auto import tqdm
bp = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
bm = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large", torch_dtype=torch.float16).to("cuda")
caps = []
for f in tqdm(reais):
    im = Image.open(f"{REAL}/{f}").convert("RGB")
    inp = bp(im, text="a photograph of", return_tensors="pt").to("cuda", torch.float16)
    out = bm.generate(**inp, max_new_tokens=40, num_beams=3)
    caps.append(bp.decode(out[0], skip_special_tokens=True))
cap_df = pd.DataFrame(dict(real=reais, legenda=caps)); cap_df.to_csv(f"{ROOT}/legendas.csv", index=False)
print(cap_df.sample(8, random_state=0).to_string())
del bm; torch.cuda.empty_cache()

In [ ]:
# 4) GeraÃ§Ã£o: alterna SDXL / SD1.5 / SDXL-Turbo. Retoma de onde parou (pula o que jÃ¡ existe).
import random, gc, torch, pandas as pd
from tqdm.auto import tqdm
if "cap_df" not in globals(): cap_df = pd.read_csv(f"{ROOT}/legendas.csv")   # legendas salvas pela celula 3
from diffusers import StableDiffusionXLPipeline, StableDiffusionPipeline, AutoPipelineForText2Image
GENS = {
  "sdxl":   dict(repo="stabilityai/stable-diffusion-xl-base-1.0", steps=30, guid=6.0, size=1024),
  "sd15":   dict(repo="stable-diffusion-v1-5/stable-diffusion-v1-5", steps=30, guid=7.5, size=512),
  "sdxlturbo": dict(repo="stabilityai/sdxl-turbo", steps=4, guid=0.0, size=512),
}
SUFIXO = ", press photograph, natural light, realistic, Brazil"
NEG = "text, watermark, logo, caption, cartoon, painting, illustration"
plano = [(r, c, list(GENS)[i % 3], 1000 + i) for i, (r, c) in enumerate(zip(cap_df.real, cap_df.legenda))]
log = []
for g in GENS:
    cfg = GENS[g]; todo = [(r, c, s) for (r, c, gg, s) in plano if gg == g and not os.path.exists(f"{FAKE}/{g}_{s}.png")]
    if not todo: continue
    if g == "sdxl": pipe = StableDiffusionXLPipeline.from_pretrained(cfg["repo"], torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    elif g == "sd15": pipe = StableDiffusionPipeline.from_pretrained(cfg["repo"], torch_dtype=torch.float16, safety_checker=None)
    else: pipe = AutoPipelineForText2Image.from_pretrained(cfg["repo"], torch_dtype=torch.float16, variant="fp16")
    pipe.to("cuda")
    for r, c, s in tqdm(todo, desc=g):
        prompt = c + SUFIXO; gen = torch.Generator("cuda").manual_seed(s)
        kw = dict(prompt=prompt, num_inference_steps=cfg["steps"], guidance_scale=cfg["guid"], generator=gen, height=cfg["size"], width=cfg["size"])
        if cfg["guid"] > 1: kw["negative_prompt"] = NEG
        img = pipe(**kw).images[0]; fn = f"{g}_{s}.png"; img.save(f"{FAKE}/{fn}")
        log.append(dict(arquivo=fn, gerador=g, repo=cfg["repo"], prompt=prompt, semente=s, passos=cfg["steps"], guidance=cfg["guid"], tamanho=cfg["size"], real_de_origem=r))
        if len(log) % 10 == 0: pd.DataFrame(log).to_csv(f"{ROOT}/geradas_log_parcial.csv", index=False)
    del pipe; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(log).to_csv(f"{ROOT}/geradas_log.csv", index=False)
print("geradas:", len(os.listdir(FAKE)))

**Depois de rodar:** abra algumas imagens de `fake_geradas/` para conferir que (a) nÃ£o tÃªm texto ou marca d'Ã¡gua e (b) nÃ£o parecem retratar uma pessoa real identificÃ¡vel. Apague as que falharem e anote no log.